# RQ2-v1 PureGeo failure diagnostics — CPU only

This notebook diagnoses why PureGeo improves the low/mid compression cliff but may degrade high widths. It consumes only already-exported RQ2-v1 tables and predictions; it does not train or load checkpoints.

## Secure checkout
Create a Kaggle secret named `github_token`. No GPU and no `KAGGLE_API_TOKEN` are required.

In [ ]:
import os, subprocess, sys, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

## Locate the completed RQ2-v1 output

In [ ]:
import importlib
import rq2_anchor_placement
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)
RQ2_INPUT_ROOT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
assert RQ2_INPUT_ROOT.exists(), f'Attach notebook output: {RQ2_INPUT_ROOT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(
    RQ2_INPUT_ROOT, '/kaggle/working/materialized-rq2-v1-diagnostics'
)
print('Validated RQ2-v1 root:', RQ2_ROOT)

## Build diagnostic tables and figures

In [ ]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
import rq2_v1_diagnostics
rq2_v1_diagnostics = importlib.reload(rq2_v1_diagnostics)
OUTPUT_DIR = Path('/kaggle/working/rq2-v1-diagnostics')
started = time.perf_counter()
result = rq2_v1_diagnostics.run_rq2_v1_diagnostics(RQ2_ROOT, OUTPUT_DIR)
print(f'Completed in {time.perf_counter() - started:.1f} seconds')
print(json.dumps(result, indent=2))

## Inspect the three diagnostic questions

In [ ]:
display(Markdown((OUTPUT_DIR / 'rq2_v1_diagnostic_report.md').read_text()))
display(Markdown('### Dense accuracy deltas and support-distance changes'))
display(pd.read_csv(OUTPUT_DIR / 'rq2_v1_accuracy_deltas.csv'))
display(Markdown('### Dense learned-projection geometry deltas'))
display(pd.read_csv(OUTPUT_DIR / 'rq2_v1_geometry_deltas.csv'))
display(Markdown('### Sample-level breadth of the effect'))
display(pd.read_csv(OUTPUT_DIR / 'rq2_v1_sample_transitions.csv'))
display(Markdown('### Artifact availability and limitation'))
display(json.loads((OUTPUT_DIR / 'rq2_v1_artifact_availability.json').read_text()))

## Figures

In [ ]:
for filename in [
    'rq2_v1_dense_accuracy_tradeoff.png',
    'rq2_v1_dense_geometry_tradeoff.png',
    'rq2_v1_geo_training_convergence.png',
]:
    path = OUTPUT_DIR / filename
    if path.is_file():
        display(Image(filename=str(path)))

## Validate and export

In [ ]:
REQUIRED = [
    'rq2_v1_dense_accuracy.csv', 'rq2_v1_dense_geometry.csv',
    'rq2_v1_accuracy_deltas.csv', 'rq2_v1_geometry_deltas.csv',
    'rq2_v1_anchor_accuracy.csv', 'rq2_v1_training_compute.csv',
    'rq2_v1_geo_training_curves.csv', 'rq2_v1_geo_training_tail_summary.csv',
    'rq2_v1_region_summary.csv', 'rq2_v1_sample_transitions.csv',
    'rq2_v1_class_accuracy_deltas.csv', 'rq2_v1_artifact_availability.json',
    'rq2_v1_dense_accuracy_tradeoff.png', 'rq2_v1_dense_geometry_tradeoff.png',
    'rq2_v1_diagnostic_report.md',
]
missing = [name for name in REQUIRED if not (OUTPUT_DIR / name).is_file()]
assert not missing, f'Missing outputs: {missing}'
bundle_path = Path('/kaggle/working/rq2-v1-diagnostics.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            bundle.write(path, path.name)
print('Download:', bundle_path)
bundle_path